# 🛣️ Caminho Mínimo de Origem Única — Bellman-Ford

O Algoritmo de **Bellman-Ford** resolve caminhos mínimos de uma única origem em digrafos ponderados que podem conter **arestas de peso negativo**. Detecta e sinaliza **ciclos de peso negativo** alcançáveis a partir da origem.

## 🎯 Problema e Modelagem
- Grafo orientado e ponderado: $G=(V,E,w)$ com $w:E\to\mathbb{R}$.
- Peso de caminho: $w(p)=\sum w(e)$.
- Distância ótima: $\delta(s,v)=\min_{p:s\leadsto v} w(p)$ ou $+\infty$ se $v$ não é alcançável.
- Objetivo: estimar $d[v]$ e predecessores $\pi[v]$ formando uma árvore (ou floresta) de caminhos mínimos.

## 🧠 Sub-estrutura Ótima
Se $p=\langle v_1, v_2, \ldots, v_k \rangle$ é um caminho mínimo de $v_1$ a $v_k$, então qualquer subcaminho $p_{ij}=\langle v_i, \ldots, v_j \rangle$ também é mínimo. Essa propriedade fundamenta os algoritmos exatos.

## ⚠️ Arestas e Ciclos Negativos
- Arestas negativas são permitidas.
- Se há um **ciclo de peso negativo** alcançável de $s$, então $\delta(s,v)=-\infty$ para vértices após o ciclo — não há menor valor bem-definido.
- Bellman-Ford detecta a presença de ciclo negativo.

## 🔧 Técnica de Relaxamento e Inicialização
- Inicializa: $d[s]=0$ e $d[v]=+\infty$ (outros), $\pi[v]=\text{NULL}$.
- Relaxa repetidamente arestas $(u,v)$: se $d[v] > d[u] + w(u,v)$, faça $d[v] \leftarrow d[u] + w(u,v)$ e $\pi[v] \leftarrow u$.
- Após $|V|-1$ iterações, se ainda há relaxamento possível, existe ciclo negativo acessível.

In [ ]:
from math import inf
from typing import Any, Dict, Tuple, List, Optional
import networkx as nx
import matplotlib.pyplot as plt

def bellman_ford(G: nx.Graph, s: Any, weight: str = 'weight') -> Tuple[Dict[Any,float], Dict[Any,Optional[Any]], bool, Optional[List[Any]]]:
    """
    Implementação do Bellman-Ford.
    Retorna (d, pi, tem_ciclo_negativo, ciclo_negativo)
      - d[v]: estimativa de distância a partir de s
      - pi[v]: predecessor
      - tem_ciclo_negativo: True se houver ciclo negativo alcançável de s
      - ciclo_negativo: lista de vértices que formam um ciclo negativo (se detectado)
    Suporta nx.DiGraph (direto). Para nx.Graph, considera-se arestas em ambos os sentidos.
    """
    d: Dict[Any,float] = {v: inf for v in G.nodes()}
    pi: Dict[Any,Optional[Any]] = {v: None for v in G.nodes()}
    d[s] = 0.0

    def edges_iter():
        if isinstance(G, nx.DiGraph):
            for u, v, data in G.edges(data=True):
                w = float(data.get(weight, 1.0))
                yield (u, v, w)
        else:
            for u, v, data in G.edges(data=True):
                w = float(data.get(weight, 1.0))
                # dois sentidos
                yield (u, v, w)
                yield (v, u, w)

    V = list(G.nodes())
    # Relaxa |V|-1 vezes
    for _ in range(len(V) - 1):
        atualizado = False
        for u, v, w in edges_iter():
            if d[u] + w < d[v]:
                d[v] = d[u] + w
                pi[v] = u
                atualizado = True
        if not atualizado:
            break

    # Verifica ciclo negativo
    for u, v, w in edges_iter():
        if d[u] + w < d[v]:
            # Reconstruir um ciclo negativo alcançável
            x = v
            # Sobe |V| passos para garantir entrar no ciclo
            for _ in range(len(V)):
                if pi[x] is None:
                    break
                x = pi[x]
            # Percorre até repetir para extrair o ciclo
            ciclo = []
            y = x
            seen = set()
            while y not in seen and y is not None:
                seen.add(y)
                ciclo.append(y)
                y = pi[y]
            # Rotaciona ciclo para iniciar em y
            if y is not None and y in ciclo:
                i = ciclo.index(y)
                ciclo = ciclo[i:]
            return d, pi, True, ciclo
    return d, pi, False, None

def reconstruir_caminho(pi: Dict[Any,Any], s: Any, v: Any) -> Optional[List[Any]]:
    if v is None:
        return None
    if v == s:
        return [s]
    if pi.get(v) is None:
        return None
    caminho = [v]
    while v != s:
        v = pi[v]
        caminho.append(v)
    caminho.reverse()
    return caminho

## 🧪 Exemplo 1 — Arestas negativas (sem ciclo negativo)

In [ ]:
G = nx.DiGraph()
G.add_weighted_edges_from([
    ('s','a', 4),
    ('s','b', 5),
    ('a','c', -2),
    ('b','c', 3),
    ('c','t', 2)
])
d, pi, neg, ciclo = bellman_ford(G, 's')
print('tem ciclo negativo?', neg, '| ciclo:', ciclo)
for v in G.nodes():
    print(v, d[v], reconstruir_caminho(pi, 's', v))

# Visualização do caminho s->t
pos = nx.spring_layout(G, seed=2)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=900, font_size=12, arrows=True, arrowsize=20)
labels = {(u,v): f'{w:.1f}' for u,v,w in G.edges(data='weight')}
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_color='darkgreen')
cam = reconstruir_caminho(pi, 's', 't')
if cam:
    path_edges = list(zip(cam, cam[1:]))
    nx.draw_networkx_edges(G, pos, edgelist=path_edges, edge_color='crimson', width=3, arrows=True, arrowsize=20)
plt.title('Caminho mínimo s→t em vermelho')
plt.show()

## 🧪 Exemplo 2 — Ciclo de peso negativo

In [ ]:
H = nx.DiGraph()
H.add_weighted_edges_from([
    ('s','a', 1),
    ('a','b', -2),
    ('b','c', -2),
    ('c','a', -2),  # ciclo negativo a→b→c→a
    ('c','t', 5)
])
d2, pi2, neg2, ciclo2 = bellman_ford(H, 's')
print('tem ciclo negativo?', neg2, '| ciclo:', ciclo2)
pos = nx.spring_layout(H, seed=3)
nx.draw(H, pos, with_labels=True, node_color='lightyellow', node_size=900, font_size=12, arrows=True, arrowsize=20)
labels = {(u,v): f'{w:.1f}' for u,v,w in H.edges(data='weight')}
nx.draw_networkx_edge_labels(H, pos, edge_labels=labels, font_color='darkgreen')
if ciclo2:
    cyc_edges = list(zip(ciclo2, ciclo2[1:]+[ciclo2[0]]))
    nx.draw_networkx_edges(H, pos, edgelist=cyc_edges, edge_color='crimson', width=3, arrows=True, arrowsize=20)
plt.title('Ciclo negativo em vermelho')
plt.show()

## ✅ Verificação opcional (NetworkX)
Para instâncias sem ciclo negativo, podemos comparar com o resultado de uma rotina do NetworkX (se disponível na sua versão) para conferir distâncias e predecessores.

In [ ]:
try:
    from networkx.algorithms.shortest_paths.weighted import single_source_bellman_ford
    dist_nx, pred_nx = single_source_bellman_ford(G, 's')
    print('Distâncias NX:', dist_nx)
except Exception as e:
    print('Verificação NetworkX indisponível nesta versão:', e)

## ⏱️ Complexidade
- Relaxamento de todas as arestas por $|V|-1$ vezes: $O(|V||E|)$.
- Verificação de ciclo negativo: $O(|E|)$.
- Total: **$O(|V||E|)$**.

## 🧩 Exercícios
1. Acrescente parada antecipada (early stopping): pare quando uma passagem não fizer atualizações.
2. Gere digrafos aleatórios com algumas arestas negativas (evitando ciclos negativos) e compare com NetworkX.
3. Implemente a reconstrução de ciclo negativo a partir de qualquer aresta relaxável depois das $|V|-1$ iterações.
4. Compare Bellman-Ford com Dijkstra em grafos sem pesos negativos (tempo e resultados).
5. Modele um exemplo de roteamento por vetor de distâncias (pequena rede) e simule o cálculo.